#SILVER - TRATAMENTO DE DADOS

### IMPORTS e CONFIGS

In [0]:
from pyspark.sql import functions as F
from pyspark.sql.types import (
    DoubleType, LongType, StringType, TimestampType
)

CATALOG = "projetos"
SCHEMA  = "crypto"

TABELA_BRONZE = f"{CATALOG}.{SCHEMA}.bronze_market_data"
TABELA_SILVER = f"{CATALOG}.{SCHEMA}.silver_market_data"

print(f"📥 Origem:  {TABELA_BRONZE}")
print(f"📤 Destino: {TABELA_SILVER}")

### Leitura da Bronze

In [0]:
from pyspark.sql import functions as F

df_bronze = spark.table(TABELA_BRONZE)

df_exploded = (
    df_bronze
    .withColumn("coin", F.explode("data"))
)

df_flat = df_exploded.select(
    "extracted_at",
    "_file_path",
    "_ingested_at",
    "coin.*"
)

df_final = df_flat.withColumn(
    "file_date",
    F.regexp_extract(F.col("_file_path"), r"(\d{4}-\d{2}-\d{2})", 1)
)

print(f"📊 Registros: {df_final.count():,}")
display(df_final)

### 2 · Seleção, renomeação e tipagem das colunas

In [0]:
df_silver = (
    df_final
    .select(
        # Identificação
        F.col("id").alias("coin_id"),
        F.col("symbol").alias("symbol"),
        F.col("name").alias("name"),

        # Preço e mercado
        F.col("current_price").cast(DoubleType()).alias("preco_usd"),
        F.col("market_cap").cast(LongType()).alias("market_cap_usd"),
        F.col("market_cap_rank").cast(LongType()).alias("ranking_market_cap"),
        F.col("total_volume").cast(LongType()).alias("volume_24h_usd"),

        # Variações
        F.col("price_change_percentage_1h_in_currency").alias("variacao_pct_1h"),
        F.col("price_change_percentage_24h").alias("variacao_pct_24h"),
        F.col("price_change_percentage_7d_in_currency").alias("variacao_pct_7d"),

        # Supply
        F.col("circulating_supply").alias("supply_circulante"),
        F.col("total_supply").alias("supply_total"),
        F.col("max_supply").alias("supply_maximo"),

        # ATH / ATL
        F.col("ath").alias("ath_usd"),
        F.col("ath_change_percentage").alias("variacao_pct_vs_ath"),
        F.col("atl").alias("atl_usd"),

        # Tempo
        F.to_timestamp("last_updated").alias("ultima_atualizacao"),
        F.to_timestamp("extracted_at").alias("extraido_em"),
        F.col("file_date").alias('data_arquivo'),
        F.current_timestamp().alias("processado_em"),
    )
)

### Remover duplicatas e grava na Silver

In [0]:
display(df_silver)

In [0]:
df_silver = df_silver.dropDuplicates(["coin_id", "extraido_em"])

(
    df_silver
    .write
    .format("delta")
    .mode("append")
    .option("mergeSchema", "true")
    .partitionBy("extraido_em")
    .saveAsTable(TABELA_SILVER)
)

print(f"✅ Silver gravada: {TABELA_SILVER}")